# Week 3, Day 2: Matplotlib II: figures that tell the truth

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w003/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Bug hunt: the exaggerated bar chart

This chart makes a 10 per cent difference look like a factor of three. Fix it so the bars start at zero, and label the y axis with its unit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(3, 3))
ax.bar(["group A", "group B"], [20, 22], color=["#c2416b", "#1f6fc7"])
ax.set_ylim(19, 22.5)
plt.show()

In [ ]:
def test_starts_at_zero():
    "the bar chart's y axis starts at zero"
    assert ax.get_ylim()[0] == 0, f"The y axis starts at {ax.get_ylim()[0]}; bars must start at zero."

def test_labelled():
    "the y axis is labelled with units"
    assert "(Hz)" in ax.get_ylabel()

test_starts_at_zero()
test_labelled()
print("All checks passed")

### Exercise: A log-scale histogram

`rates` holds 400 firing rates. Make `bins` with `np.logspace` (30 bins from 0.01 Hz to 1000 Hz), draw a histogram with them on `ax`, and set the x axis to a log scale.

In [ ]:
rng = np.random.default_rng(0)
rates = np.exp(rng.normal(np.log(3), 1.1, 400))

fig, ax = plt.subplots(figsize=(5, 3))
bins = ...
# histogram and log x axis here
ax.set_xlabel("firing rate (Hz)"); ax.set_ylabel("number of neurons")
plt.show()

In [ ]:
import numpy as np
def test_log_axis():
    "the x axis is logarithmic"
    assert ax.get_xscale() == "log", "Use ax.set_xscale('log')."

def test_log_bins():
    "bins are evenly spaced in log space"
    assert np.allclose(np.diff(np.log10(bins)), np.diff(np.log10(bins))[0]), "Make bins with np.logspace."
    assert bins[0] <= rates.min() and bins[-1] >= rates.max(), "The bins should cover all the data."

test_log_axis()
test_log_bins()
print("All checks passed")

### Exercise: A raster with eventplot

`trains` is a list of 30 arrays of spike times (seconds). Draw them as a raster on `ax` with `ax.eventplot(trains, color='k', linelengths=0.8)`, label the x axis `time (s)` and the y axis `trial`, and mark the stimulus onset at 0.2 s with `ax.axvline`.

In [ ]:
rng = np.random.default_rng(1)
def one_trial():
    t = np.arange(0, 1, 0.001)
    rate = 5 + 60 * ((t > 0.25) & (t < 0.45))
    return t[rng.random(t.size) < rate * 0.001]
trains = [one_trial() for _ in range(30)]

fig, ax = plt.subplots(figsize=(6, 3))
# raster here
plt.show()

In [ ]:
def test_rows():
    "one row of ticks per trial"
    assert len(ax.collections) == len(trains), f"Expected {len(trains)} rows, found {len(ax.collections)}."

def test_labels():
    "axes are labelled"
    assert "(s)" in ax.get_xlabel() and ax.get_ylabel(), "Label time (s) and trials."

test_rows()
test_labels()
print("All checks passed")

### Exercise: A population heatmap

`pop` has shape (40 neurons, 100 time bins). Show it with `ax.imshow(pop, aspect='auto', cmap='viridis', extent=[0, 1, 40, 0])`, add a colour bar labelled `rate (Hz)`, and label both axes.

In [ ]:
rng = np.random.default_rng(2)
peak = np.sort(rng.uniform(0.1, 0.9, 40))
tt = np.linspace(0, 1, 100)
pop = 2 + 30 * np.exp(-(tt[None, :] - peak[:, None]) ** 2 / (2 * 0.05 ** 2)) + rng.normal(0, 1.5, (40, 100))

fig, ax = plt.subplots(figsize=(5, 3.5))
im = ...
# colour bar and labels
plt.show()

In [ ]:
def test_image():
    "the heatmap uses viridis"
    im = ax.images[0]
    assert im.get_cmap().name == "viridis", "Use cmap='viridis'."

def test_colorbar():
    "there is a colour bar with a unit"
    assert len(fig.axes) == 2, "Add a colour bar with fig.colorbar(im, ax=ax)."
    cb_ax = [a for a in fig.axes if a is not ax][0]
    assert "(Hz)" in cb_ax.get_ylabel(), "Label the colour bar, for example 'rate (Hz)'."

test_image()
test_colorbar()
print("All checks passed")

In [ ]:
rng = np.random.default_rng(4)
groups = {"smooth": rng.normal(20, 4, 30),
          "two clusters": np.r_[rng.normal(15, 1, 15), rng.normal(25, 1, 15)],
          "one outlier": np.r_[rng.normal(18.5, 1.5, 29), 60]}
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for i, (name, x) in enumerate(groups.items()):
    axes[0].bar(i, x.mean(), yerr=x.std(ddof=1) / np.sqrt(len(x)), capsize=4, color="0.7")
    axes[1].plot(np.full(len(x), i) + rng.uniform(-0.15, 0.15, len(x)), x, "o", ms=4, alpha=0.7)
    axes[1].plot([i - 0.25, i + 0.25], [x.mean()] * 2, "k-", lw=2)
for ax, t in zip(axes, ["bars hide it", "points show it"]):
    ax.set_xticks(range(3), groups.keys()); ax.set_title(t)
axes[0].set_ylabel("firing rate (Hz)")
plt.tight_layout(); plt.show()